In [44]:
#bringing in dependencies...

#note to self: have <naive.py>, <PythiaEventsBatchTest.root> on hand!

!pip install uproot

import naive

InputJets = naive.JetReader("PythiaEventsBatchTest.root",'pdEventTree')

In [45]:
##### POPULATING GHOST JETS #####

# phi bounds are obvious: 0 to 2pi
# ? How do I decide bounds of y?
# For simplicity for now, I'll settle for max, min

ymin = min([jet.y for jet in InputJets])
ymax = max([jet.y for jet in InputJets])

print(f"y_min: {ymin}")
print(f"y_max: {ymax}")

y_min: -8.514232250012876
y_max: 9.136680552668757


In [46]:
import numpy as np
pi = float(np.pi)
pi

3.141592653589793

In [47]:
import time
import statistics

def TimingMeasurement(inputjets, ClusterFunc, *args, **kwargs):

  """
  Measures how much time it takes to cluster a set of jets.

  returns N, time_result, dt
    with N = number of initial jets clustered (including ghosts),
    time_result = average time elapsed in seconds,
    and dt = standard deviation from five trials

  """

  t_list = []

  for count in range(5):

    print(f"Trial {count} begins...")
    start = time.perf_counter()
    outhistory, outjets = ClusterFunc(*args, **kwargs)
    end = time.perf_counter()

    t_list.append(end-start)
    print(f"Trial {count} complete: reading {end-start} seconds")

  dt = statistics.stdev(t_list)
  time_result = statistics.mean(t_list)



  return len(inputjets), time_result, dt

In [48]:
# How many ghost jets do you want? N, with N = ny*nphi

ny = 10
nphi = 10

N = ny*nphi

print(f"N: {N}")



N: 100


In [49]:
### Bringing in time module, testing it out...

In [50]:
print(f"Populating {N} ghost jets...")

import time

print("time test:")

start1 = time.perf_counter_ns()
InputJetsGhost = naive.GhostJetPopulator(InputJets, ymin, ymax, ny, 0, 2*pi, nphi)
end1 = time.perf_counter_ns()

print(f"Completed in {end1-start1} nanoseconds, or {(end1-start1)*(10**(-9))} s")

Populating 100 ghost jets...
time test:
Completed in 5165465 nanoseconds, or 0.005165465 s


In [51]:
### just gonna test how this workflow goes,
### then replicate it in a callable function

t_data = []
N_data = []

start = time.perf_counter_ns()
outhistory_test, outjets_test = naive.gen_clustering(-1, InputJets, 0.4)
end = time.perf_counter_ns()

t_data.append((end-start)*(10**(-9))) #take result in nanosecs, convert to seconds
N_data.append(N)

print(f"Clustered {N} initial jets -- Completed in {end-start} nanoseconds, or {(end-start)*(10**(-9))} s")






Clustered 100 initial jets -- Completed in 75224497689 nanoseconds, or 75.224497689 s


In [52]:
help(naive.gen_clustering)

Help on function gen_clustering in module naive:

gen_clustering(mode, j_initial, R_0, thresholding=False, pT_threshold=0.0, eta_threshold=0.0)
    Function for clustering an input set of jets, using anti-kT clustering algorithm,
    to return a set of final-state jets.

    Parameters:
      mode: The clustering mode to use.
        1: kT-algorithm
        0: Cambridge-Aachen
        (-1): anti-kT algorithm

      j_initial: List of initial jets,
      R_0: the jet radius parameter,

      thresholding: Boolean, configurable by user in case they wish to apply jet pT and eta thresholds; off by default
      pT_threshold: minimum jet pT,
      eta_threshold: minimum jet eta,

    Returns: clustering_history, final_jets

      clustering_history: list of tuples--('final'/'merge', ids of jets being promoted)
      final_jets: List of final-state jets



In [53]:
n_res, t_res, dt_res = TimingMeasurement(inputjets=InputJets, j_initial=InputJets, ClusterFunc=naive.gen_clustering, mode=-1, R_0=0.4)

Trial 0 begins...
Trial 0 complete: reading 76.00772054700064 seconds
Trial 1 begins...
Trial 1 complete: reading 69.4350470520003 seconds
Trial 2 begins...
Trial 2 complete: reading 69.97326906600028 seconds
Trial 3 begins...
Trial 3 complete: reading 70.70191494699975 seconds
Trial 4 begins...
Trial 4 complete: reading 70.89520949199868 seconds


In [56]:
print(f"N: {n_res} jets clustered")
print(f"Time: {t_res} +/- {dt_res} seconds")

N: 243 jets clustered
Time: 71.40263222079993 +/- 2.639592366275294 seconds


In [58]:
##### Now to do this for a wider spectrum of N #####


#collecting N,t,dt data using TimingMeasurement
# need to input:
# InputJets -- list of real input InputJets
# ny, nphi -- linear density of ghost jets in y and phi directions respectively
# for now just letting ny = nphi

nlist = [5,6,7,8,9,10]

N_data = []
t_data = []
dt_data = []

runcount = len(nlist)

for _n in nlist:

  InputJetsGhosted = naive.GhostJetPopulator(InputJets, ymin, ymax, _n, 0, 2*pi, _n)

  n_result, t_result, dt_result = TimingMeasurement(inputjets=InputJetsGhosted, j_initial=InputJetsGhosted, ClusterFunc=naive.gen_clustering, mode=-1, R_0=0.4)

  N_data.append(n_result)
  t_data.append(t_result)
  dt_data.append(dt_result)

  print(f"{n_result} clustered in {t_result} +/- {dt_result} seconds")

Trial 0 begins...
Trial 0 complete: reading 126.07441154199842 seconds
Trial 1 begins...
Trial 1 complete: reading 128.56294241299838 seconds
Trial 2 begins...
Trial 2 complete: reading 126.70903289600028 seconds
Trial 3 begins...
Trial 3 complete: reading 127.08342084200012 seconds
Trial 4 begins...
Trial 4 complete: reading 134.87792684000124 seconds
293 clustered in 128.66154690659968 +/- 3.5933676598242026 seconds
Trial 0 begins...


KeyboardInterrupt: 